In [2]:
from pathlib import Path

import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from nre.preprocess import preprocess_df
from nre.network_connectivity import ConnectivityUnit, ton_iot_conn_param_specs
from nre.analyze_nf_dataset import nre_classification, flow_based_classification
from nre.classification_tools import plot_roc_curves
from nre.validation_tools import validate_method

from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB


# CIC-IDS-2017

In [4]:

# CIC-IDS-2017
file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv'  # 'Monday-WorkingHours.pcap_ISCX.csv' #  Wednesday-workingHours.pcap_ISCX.csv
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Morning.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'

# NF-ToN-IoT
file_addr = r'../../NF-ToN-IoT/NF-ToN-IoT-v3.csv'
#file_addr = r'..\..\NF-ToN-IoT\NF-ToN-IoT-v3.csv'

file_addr = Path(file_addr)

# Definitions
date_col = 'FLOW_START_MILLISECONDS' # 'FLOW_START_MILLISECONDS' # ' Timestamp'
label_col = 'Label' # 'Label' # ' Label'
benign_label = '0'  # '0' # 'BENIGN'
src_id_col = 'IPV4_SRC_ADDR' # ' Source IP'
dst_id_col = 'IPV4_DST_ADDR' # ' Total Backward Packets'
src_feature_col = 'OUT_PKTS' # ' Total Fwd Packets'
dst_feature_col = 'IN_PKTS' # ' Total Backward Packets'
conn_param = 'NPS'
conn_param_specs = ton_iot_conn_param_specs
feat_cols = ('OUT_PKTS', 'IN_PKTS')

In [5]:
#df_cic = pd.read_csv(file_addr, header=0, encoding='cp1252')
df_iot = pd.read_csv(file_addr, sep=',', encoding='utf-8')

#df = preprocess_df(df_cic, date_col=' Timestamp')
df_whole = preprocess_df(df_iot, date_col=date_col, ds_type='ton-iot')

print(df_whole.shape)

(27520260, 55)


# Train - Test Split
Use the same split (seed) in experiment section

In [10]:
seed = 138
test_size = 0.33

ind_co = int(df_whole.shape[0] * (1 - test_size))
df_train, df_test = df_whole.iloc[:ind_co, :], df_whole.iloc[ind_co:, :]

In [11]:
df_train.shape, df_test.shape

((18438574, 55), (9081686, 55))

In [12]:
np.unique(df_train[label_col].values, return_counts= True)

(array([0, 1], dtype=object), array([13221986,  5216588]))

In [13]:
type(df_train[label_col][0])

int

# Network Scope 

## CIC-IDS-2017

In [6]:
with open(r'saves\victim_net.pickle', 'rb') as handle:
    entity_names = pickle.load(handle) 
len(entity_names)

13

In [7]:
with open(r'saves\partitioned_nodes_141.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
len(subnet_names)

141

## NF-ToN-IoT

In [9]:
with open(r'saves/ton_iot_freq_1000.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
entity_names = subnet_names.copy()
len(subnet_names)

15

In [14]:
with open(r'saves/partitioned_nodes_iot_261.pickle', 'rb') as handle: #105
    subnet_names = pickle.load(handle) 
entity_names = subnet_names.copy()
len(subnet_names)

261

# Validation

In [71]:
import importlib
import nre.validation_tools

importlib.reload(nre.validation_tools)
from nre.validation_tools import *

In [72]:
import importlib
import nre.kalman_network_tools

importlib.reload(nre.kalman_network_tools)
from nre.kalman_network_tools import *

In [92]:
import importlib
import nre.analyze_nf_dataset

importlib.reload(nre.analyze_nf_dataset)
from nre.analyze_nf_dataset import *

In [18]:

def model_nre(df_train, ml_models, **kwargs):
    return nre_classification(df_train, ml_models, entity_names=subnet_names, standardize=True, **kwargs)  # entity_names=subnet_names

def model_fb(df_train, ml_models, **kwargs):
    return flow_based_classification(df_train, ml_models, entity_names=None, standardize=True, **kwargs)  # entity_names=entity_names

val_size = 0.25
n_train = df_train.shape[0]
n_val = int(n_train * val_size)

df_val = df_train.iloc[n_train-n_val:, :]

# CIC-IDS-2017
"""
param_list = [{'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 45, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 720, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.6, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.3, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 2.4, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 4.8, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10}
              ]
"""

# ToN-IoT
okwargs = {'date_col':date_col, 'src_id_col': src_id_col, 'dst_id_col': dst_id_col, 'conn_param':conn_param, 'window_type':'conn',
          'label_col':label_col, 'conn_param_specs':ton_iot_conn_param_specs, 'benign_label': benign_label, 'seed':345, 'verbose':False,
          'labelling_opt': 'majority'}
param_list = [{'n_graph': 10_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 8_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 2_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':50, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':25, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':200, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':400, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1, **okwargs},
              {'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3, **okwargs},
              {'n_graph': 5_000, 'conn_size':100, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10, **okwargs}
              ]

param_list_fb = [{**params, 'feat_cols': feat_cols} for params in param_list]

auc_scores_nre = validate_method(df_train, df_val, model_nre, param_list)
#auc_scores_fb = validate_method(df_train, df_val, model_fb, param_list_fb)

  0%|                                                                                                                                                                                                         | 0/12 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:2458: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_ranking.py:1146: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\bayer\PycharmProjects\NRE\v

[-1  1]


 17%|████████████████████████████████▏                                                                                                                                                                | 2/12 [00:45<03:49, 22.97s/it]

[-1  1]


C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\

...

[-1  1]


C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:2458: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_ranking.py:1146: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

[-1  1]


C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:2458: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_ranking.py:1146: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

[-1  1]


In [20]:
auc_scores_nre

[({'n_graph': 10000,
   'conn_size': 100,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'conn',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_feature_co

In [94]:
auc_scores_fb

[({'n_graph': 10000,
   'conn_size': 100,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1,
   'date_col': 'FLOW_START_MILLISECONDS',
   'src_id_col': 'IPV4_SRC_ADDR',
   'dst_id_col': 'IPV4_DST_ADDR',
   'conn_param': 'NPS',
   'window_type': 'conn',
   'label_col': 'Label',
   'conn_param_specs': {'Activation': {'method': 'activation'},
    'Bytes Sent': {'method': 'total',
     'src_feature_col': 'OUT_BYTES',
     'dst_feature_col': 'IN_BYTES'},
    'Bytes Received': {'method': 'total',
     'src_feature_col': 'IN_BYTES',
     'dst_feature_col': 'OUT_BYTES'},
    'Flow Duration': {'method': 'total',
     'src_feature_col': 'FLOW_DURATION_MILLISECONDS',
     'dst_feature_col': 'FLOW_DURATION_MILLISECONDS'},
    'NPS': {'method': 'total',
     'src_feature_col': 'OUT_PKTS',
     'dst_feature_col': 'IN_PKTS'},
    'NPR': {'method': 'total',
     'src_feature_col': 'IN_PKTS',
     'dst_feature_col': 'OUT_PKTS'},
    'Port Number': {'method': 'mode',
     'src_feature_co

In [36]:
np.where(df_train[label_col] == 1)

(array([     1,      2,      3, ..., 339798, 339799, 339805], dtype=int64),)

In [12]:
res = (auc_scores_nre, auc_scores_fb)
with open(r'saves\validation_curves_141.pickle', 'wb') as handle:
    pickle.dump(res, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [95]:
res = (auc_scores_nre, auc_scores_fb)
with open(r'saves\validation_curves_iot_15.pickle', 'wb') as handle:
    pickle.dump(res, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [10]:
with open(r'saves\validation_curves_iot_15.pickle', 'rb') as handle:
    auc_scores_nre, auc_scores_fb = pickle.load(handle) 

In [13]:
auc_scores_nre

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9333333333333333),
 ({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9541666666666667),
 ({'t_graph': 45,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.996011964107677),
 ({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.8636363636363636),
 ({'t_graph': 720,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.8333333333333334),
 ({'t_graph': 180,
   'sync_window_size': 0.6,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.942857142857143),
 ({'t_graph': 180,
   'sync_window_size': 0.3,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.942857142857143),
 ({'t_graph': 180,
   'sync_window_size

In [14]:
auc_scores_fb

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.6625000000000001),
 ({'t_graph': 45,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.576271186440678),
 ({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7727272727272727),
 ({'t_graph': 720,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.6666666666666666),
 ({'t_graph': 180,
   'sync_window_size': 0.6,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 180,
   'sync_window_size': 0.3,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 180,
   'sync_window_si

In [18]:
param_list = [{'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1}]
auc_scores_nre = validate_model(df_train, df_val, model_nre, param_list)
auc_scores_nre

  0%|                                                                                                                          | 0/1 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:23<00:00, 83.35s/it]


[({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9541666666666667)]

In [19]:
param_list = [{'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1}]
auc_scores_fb = validate_model(df_train, df_val, model_fb, param_list)
auc_scores_fb

  0%|                                                                                                                          | 0/1 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:15<00:00, 15.56s/it]


[({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7727272727272727)]